In [ ]:
# Import necessary libraries
import tensorflow as tf
from tensorflow.keras.applications import VGG16, VGG19
from tensorflow.keras.layers import Dense, Flatten, Conv2D, MaxPooling2D, concatenate, Input, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, confusion_matrix, classification_report
import os
import random
import shutil
import numpy as np
import pandas as pd

# Mount Google Drive to access the dataset
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Define original and reduced train directory paths in Google Drive
original_train_dir = "/content/drive/MyDrive/COVID-19_Radiography_Dataset/train"
reduced_train_dir = "/content/drive/MyDrive/COVID-19_Radiography_Dataset/reduced_train"

# Define classes to include and the percentage to reduce
classes_to_include = ['COVID', 'Normal']

# Create reduced directory if it doesn't exist
if not os.path.exists(reduced_train_dir):
    os.makedirs(reduced_train_dir)

# Select a percentage of images for each class and copy to reduced_train_dir
for class_name in classes_to_include:
    class_path = os.path.join(original_train_dir, class_name)
    target_class_path = os.path.join(reduced_train_dir, class_name)
    os.makedirs(target_class_path, exist_ok=True)
    images = os.listdir(class_path)
    selected_images = random.sample(images, len(images))
    for image in selected_images:
        shutil.copy(os.path.join(class_path, image), os.path.join(target_class_path, image))

img_size = (224, 224)
batch_size = 8

# Enhanced Data Augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.3,
    height_shift_range=0.3,
    shear_range=0.3,
    zoom_range=0.3,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest',
    validation_split=0.2
)

# Creating the train and validation generators
train_generator = train_datagen.flow_from_directory(
    reduced_train_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='binary',
    subset='training'
)

validation_generator = train_datagen.flow_from_directory(
    reduced_train_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='binary',
    subset='validation'
)

# More Complex Model Architecture
input_layer = Input(shape=(224, 224, 3))
vgg16_base = VGG16(weights='imagenet', include_top=False)
vgg19_base = VGG19(weights='imagenet', include_top=False)
vgg16_base.trainable = False
vgg19_base.trainable = False

vgg16_features = vgg16_base(input_layer)
vgg19_features = vgg19_base(input_layer)

combined_features = concatenate([vgg16_features, vgg19_features])
x = Conv2D(128, (3, 3), activation='relu', padding='same')(combined_features)
x = BatchNormalization()(x)
x = MaxPooling2D(pool_size=(2, 2))(x)
x = Dropout(0.5)(x)

atrous1 = Conv2D(64, (3, 3), dilation_rate=1, activation='relu', padding='same')(x)
atrous2 = Conv2D(64, (3, 3), dilation_rate=2, activation='relu', padding='same')(atrous1)
atrous3 = Conv2D(64, (3, 3), dilation_rate=4, activation='relu', padding='same')(atrous2)

x = BatchNormalization()(atrous3)
x = MaxPooling2D(pool_size=(2, 2))(x)
x = Dropout(0.5)(x)

x = Flatten()(x)
x = Dense(512, activation='relu')(x)
x = Dropout(0.5)(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)

output = Dense(1, activation='sigmoid')(x)

final_model = Model(inputs=input_layer, outputs=output)
final_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001), loss='binary_crossentropy', metrics=['accuracy'])

# Train the model
history = final_model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // batch_size,
    validation_data=validation_generator,
    validation_steps=validation_generator.samples // batch_size,
    epochs=50
)

# Test data processing
test_dir = "/content/drive/MyDrive/COVID-19_Radiography_Dataset/test"

test_datagen = ImageDataGenerator(rescale=1./255)
test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='binary',
    shuffle=False
)

# Predict on the test data
y_true = test_generator.classes
y_pred_prob = final_model.predict(test_generator)

# Convert probabilities to binary predictions using a threshold
threshold = 0.5
y_pred = (y_pred_prob.ravel() >= threshold).astype(int)

# Calculate test accuracy
test_accuracy = np.mean(y_pred == y_true)
print(f"Test Accuracy: {test_accuracy:.2%}")

# Save true labels and predicted probabilities to CSV
results_df = pd.DataFrame({
    'True Labels': y_true,
    'Predicted Probabilities': y_pred_prob.ravel(),
    'Predicted Labels': y_pred
})

results_df.to_csv('/content/drive/MyDrive/predictions_results.csv', index=False)

# Download the CSV file
from google.colab import files
files.download('/content/drive/MyDrive/predictions_results.csv')


In [ ]:

final_model.save('/content/drive/MyDrive/covid_detection_model.keras')
print("Model saved successfully in .keras format.")


In [ ]:
from tensorflow.keras.models import load_model
import numpy as np
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from google.colab import drive
drive.mount('/content/drive',force_remount=True)
loaded_model = load_model('/content/drive/MyDrive/covid_detection_model.keras')

print("Model loaded successfully.")

In [ ]:
def preprocess_image(image_path, img_size=(224, 224)):

    img = load_img(image_path, target_size=img_size)
    img_array = img_to_array(img)
    img_array = img_array / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    return img_array

def predict_image(image_path):

    img_array = preprocess_image(image_path)
    prediction = loaded_model.predict(img_array)
    if prediction[0][0] > 0.5:
        print(f"The model predicts the image is Normal.")
    else:
        print(f"The model predicts the image is affected with COVID-19.")

image_path =image_path = "/content/drive/MyDrive/COVID-19_Radiography_Dataset/train/COVID/COVID-99.png"
predict_image(image_path)


In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder
import numpy as np

def extract_features(model, generator):
    features = model.predict(generator)
    labels = generator.classes
    return features, labels

train_features, train_labels = extract_features(loaded_model, train_generator)
test_features, test_labels = extract_features(loaded_model, validation_generator)

train_features = train_features.reshape((train_features.shape[0], -1))
test_features = test_features.reshape((test_features.shape[0], -1))

label_encoder = LabelEncoder()
train_labels_encoded = label_encoder.fit_transform(train_labels)
test_labels_encoded = label_encoder.transform(test_labels)

svm_model = SVC(kernel='linear')
svm_model.fit(train_features, train_labels_encoded)

svm_predictions = svm_model.predict(test_features)
svm_accuracy = accuracy_score(test_labels_encoded, svm_predictions)
print(f"SVM Test Accuracy: {svm_accuracy * 100:.2f}%")
target_names = [str(label) for label in label_encoder.classes_]
report = classification_report(test_labels_encoded, svm_predictions, target_names=target_names)
print("Classification Report for SVM:")
print(report)

test_loss, test_acc = final_model.evaluate(validation_generator, verbose=0)

cnn_accuracy = test_acc * 100

print(f"CNN Test Accuracy: {cnn_accuracy:.2f}%")

if svm_accuracy * 100 > cnn_accuracy:
    print("The SVM model outperformed the CNN model.")
else:
    print("The CNN model outperformed the SVM model.")


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder
import numpy as np

def extract_features(model, generator):
    features = model.predict(generator)
    labels = generator.classes
    return features, labels

train_features, train_labels = extract_features(final_model, train_generator)
test_features, test_labels = extract_features(final_model, validation_generator)

train_features = train_features.reshape((train_features.shape[0], -1))
test_features = test_features.reshape((test_features.shape[0], -1))
label_encoder = LabelEncoder()
train_labels_encoded = label_encoder.fit_transform(train_labels)
test_labels_encoded = label_encoder.transform(test_labels)

logistic_model = LogisticRegression(max_iter=1000)
logistic_model.fit(train_features, train_labels_encoded)
logistic_predictions = logistic_model.predict(test_features)

logistic_accuracy = accuracy_score(test_labels_encoded, logistic_predictions)
print(f"Logistic Regression Test Accuracy: {logistic_accuracy * 100:.2f}%")

target_names = [str(label) for label in label_encoder.classes_]
report = classification_report(test_labels_encoded, logistic_predictions, target_names=target_names)
print("Classification Report for Logistic Regression:")
print(report)


test_loss, test_acc = final_model.evaluate(validation_generator, verbose=0)

cnn_accuracy = test_acc * 100

print(f"CNN Test Accuracy: {cnn_accuracy:.2f}%")
if logistic_accuracy * 100 > cnn_accuracy:
    print("The Logistic Regression model outperformed the CNN model.")
else:
    print("The CNN model outperformed the Logistic Regression model.")


In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder
import numpy as np

def extract_features(model, generator):
    features = model.predict(generator)
    labels = generator.classes
    return features, labels

train_features, train_labels = extract_features(final_model, train_generator)
test_features, test_labels = extract_features(final_model, validation_generator)

train_features = train_features.reshape((train_features.shape[0], -1))
test_features = test_features.reshape((test_features.shape[0], -1))

label_encoder = LabelEncoder()
train_labels_encoded = label_encoder.fit_transform(train_labels)
test_labels_encoded = label_encoder.transform(test_labels)

naive_bayes_model = GaussianNB()
naive_bayes_model.fit(train_features, train_labels_encoded)

naive_bayes_predictions = naive_bayes_model.predict(test_features)

naive_bayes_accuracy = accuracy_score(test_labels_encoded, naive_bayes_predictions)
print(f"Naive Bayes Test Accuracy: {naive_bayes_accuracy * 100:.2f}%")

target_names = [str(label) for label in label_encoder.classes_]
report = classification_report(test_labels_encoded, naive_bayes_predictions, target_names=target_names)
print("Classification Report for Naive Bayes:")
print(report)


test_loss, test_acc = final_model.evaluate(validation_generator, verbose=0)
cnn_accuracy = test_acc * 100

print(f"CNN Test Accuracy: {cnn_accuracy:.2f}%")
if naive_bayes_accuracy * 100 > cnn_accuracy:
    print("The Naive Bayes model outperformed the CNN model.")
else:
    print("The CNN model outperformed the Naive Bayes model.")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, f1_score, classification_report

y_true = test_generator.classes
y_pred = (final_model.predict(test_generator) > 0.5).astype(int).flatten()

conf_matrix = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=test_generator.class_indices.keys(), yticklabels=test_generator.class_indices.keys())
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.show()

f1 = f1_score(y_true, y_pred)
print(f"F1 Score: {f1:.2f}")

report = classification_report(y_true, y_pred, target_names=test_generator.class_indices.keys())
print("Classification Report:")
print(report)

In [ ]:
import matplotlib.pyplot as plt

def plot_train_accuracy_loss(history):
    plt.figure(figsize=(12, 6))

    # Plot training accuracy
    plt.plot(history.history['accuracy'], label='Train Accuracy', color='blue')

    # Plot training loss
    plt.plot(history.history['loss'], label='Train Loss', color='orange')

    # Set labels and title
    plt.xlabel('Epoch', labelpad=15)
    plt.ylabel('Metrics', labelpad=15)
    plt.title('Training Accuracy and Loss')
    plt.legend(loc='upper right')

    # Show the plot
    plt.tight_layout()
    plt.show()

# Call the function to plot training accuracy and loss
plot_train_accuracy_loss(history)
